# ResNet18 Edge Deployment with Comprexx

This notebook walks through compressing a ResNet18 for edge deployment:

1. Profile the original model
2. Fuse Conv+BN layers (free compression)
3. Prune 40% of filters by L1 norm
4. Quantize to INT8 via dynamic PTQ
5. Benchmark before/after latency
6. Export to ONNX

Install: `pip install "comprexx[onnx]"`

In [ ]:
import os

import torch

import comprexx as cx

## 1. Load and profile the model

In [ ]:
model = torch.hub.load("pytorch/vision", "resnet18", weights=None)
model.eval()

input_shape = (1, 3, 224, 224)
profile = cx.analyze(model, input_shape)
print(profile.summary())

## 2. Build the compression pipeline

Three stages, applied in order:
- **Operator fusion**: fold BatchNorm into Conv2d. Zero accuracy cost, fewer params.
- **Structured pruning**: zero out the least important 40% of conv filters globally.
- **PTQ dynamic**: quantize Linear layers to INT8 at runtime.

In [ ]:
pipeline = cx.Pipeline([
    cx.stages.OperatorFusion(),
    cx.stages.StructuredPruning(sparsity=0.4, criteria="l1_norm", scope="global"),
    cx.stages.PTQDynamic(),
])

result = pipeline.run(model, input_shape=input_shape)
print(result.report.summary())

## 3. Verify the compressed model still works

In [ ]:
compressed = result.model

with torch.no_grad():
    x = torch.randn(*input_shape)
    out_original = model(x)
    out_compressed = compressed(x)

print(f"Original output shape:   {out_original.shape}")
print(f"Compressed output shape: {out_compressed.shape}")
print(f"Max abs difference:      {(out_original - out_compressed).abs().max():.4f}")

## 4. Benchmark inference latency

In [ ]:
comparison = cx.compare_benchmarks(
    model, compressed,
    input_shape=input_shape,
    warmup=10,
    iters=50,
)
print(comparison.summary())

## 5. Export to ONNX

The exporter runs `torch.onnx.export`, validates the output against PyTorch, and writes a manifest with compression metadata.

In [ ]:
exporter = cx.ONNXExporter()
exporter.export(
    result.model,
    input_shape=input_shape,
    output_path="resnet18_compressed.onnx",
)

onnx_size = os.path.getsize("resnet18_compressed.onnx") / 1e6
print(f"ONNX file size: {onnx_size:.1f} MB")

## 6. Same pipeline as a YAML recipe

You can define this exact pipeline in YAML and run it from the CLI.

In [ ]:
recipe_yaml = """
name: resnet18-edge
description: Pruned and quantized ResNet18 for edge deployment

stages:
  - technique: operator_fusion

  - technique: structured_pruning
    sparsity: 0.4
    criteria: l1_norm
    scope: global

  - technique: ptq_dynamic
    format: int8
"""

print(recipe_yaml)
print("# Save as resnet18-edge.yaml, then run:")
print(
    "# comprexx compress torchvision.models.resnet18"
    " --recipe resnet18-edge.yaml --input-shape 1,3,224,224"
)